# Admission-Only Multi-Task Learning After Myocardial Infarction
### Predicting Mortality, Cause of Death, and Complications

This notebook contains the experiments reported in the paper **"Admission-Only Multi-Task Learning After Myocardial Infarction: Predicting Mortality, Cause of Death, and Complications."**

It reproduces:
1. **Section 4.1 (Tables 2–3):** comparison with prior work using a parallel classical-ML pipeline and the multitask FT-Transformer with admission-only features.
2. **Section 4.2 / Experiment 1 (Table 4, Figure 2):** time-window ablation across admission, 24h, 48h, and 72h information.
3. **Section 4.3 / Experiment 2 (Table 5, Figure 3):** hard-gated, continuous hierarchical, and flat 8-class output formulations.
4. **Section 4.4 / Experiment 3 (Tables 6–9, Figure 4):** permutation-SHAP interpretability across all four time windows.

**Data are not included in this repository.** Download `MI.data` from the UCI Machine Learning Repository and place it at `data/MI.data` before running the notebook.

The notebook can be run locally or in Google Colab. A GPU is recommended for the full experiment suite.


## 1. Setup and imports

In [ ]:
# =========================
# Setup
# =========================
!pip -q install -U scikit-learn scipy seaborn iterative-stratification

import math, os, json, random, warnings, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, brier_score_loss, log_loss, roc_curve
)
from scipy.optimize import minimize_scalar
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from itertools import combinations

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Paper-wide visualization theme
ARCH_BLUE = "#0024FD"
ARCH_PURPLE = "#461BB5"
ARCH_ORANGE = "#FE4B0D"
ARCH_GREEN = "#068245"
ARCH_RED = "#FE0000"
ARCH_GRAY = "#5B6472"

ARCH_PALETTE = [ARCH_BLUE, ARCH_PURPLE, ARCH_ORANGE, ARCH_GREEN, ARCH_RED]
ARCH_BLUE_CMAP = sns.light_palette(ARCH_BLUE, as_cmap=True)
ARCH_PURPLE_CMAP = sns.light_palette(ARCH_PURPLE, as_cmap=True)
ARCH_GREEN_CMAP = sns.light_palette(ARCH_GREEN, as_cmap=True)
ARCH_RED_CMAP = sns.light_palette(ARCH_RED, as_cmap=True)
ARCH_DIVERGING_CMAP = sns.blend_palette([ARCH_BLUE, "#FFFFFF", ARCH_RED], as_cmap=True)

sns.set_palette(ARCH_PALETTE)
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "figure.titlesize": 16,
})


## 2. Paths and configuration

In [ ]:
# =========================
# Paths and configuration
# =========================

# Repository-relative paths. The dataset is intentionally not committed.
DATASET_PATH = os.path.join("data", "MI.data")
SAVE_DIR = os.path.join("results", "mi_encoder_sharing_ablation")
os.makedirs(SAVE_DIR, exist_ok=True)

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Could not find {DATASET_PATH}. Download the UCI MI.data file and "
        "place it at data/MI.data before running the notebook."
    )

print("Dataset:", DATASET_PATH)
print("Output directory:", SAVE_DIR)

P = {
    "n_features": 102,
    "n_complications": 11,
    "n_causes": 7,
    "d_token": 192,
    "n_heads": 2,
    "n_layers": 1,
    "d_ffn_factor": 2,
    "attn_dropout": 0.3,
    "ffn_dropout": 0.4,
    "head_dropout": 0.1,
    "res_dropout": 0.1,
    "lr": 0.0002890238693888574,
    "weight_decay": 0.004503312791376398,
    "batch_size": 128,
    "w_mort": 1.0,
    "w_cause": 0.4,
    "w_comp": 2.0,
    "mort_pos_weight": 2.0,
    "comp_cap": 10.0,
    "cause_weight_cap": 10.0,
    "patience": 19,
    "epochs": 150,
}

COMP_NAMES = [
    "Atrial fibrillation", "Supravent. tachycardia", "Ventricular tachycardia",
    "Ventricular fibrillation", "3rd-degree AV block", "Pulmonary edema",
    "Myocardial rupture", "Dressler syndrome", "Chronic heart failure",
    "Recurrent MI", "Post-infarction angina",
]

CAUSE_NAMES = [
    "Cardiogenic shock", "Pulmonary edema", "Myocardial rupture",
    "Progress. HF", "Thromboembolic", "Ventricular fib.", "Unknown/other",
]

CONDITIONS = ["full"]
CONDITION_LABELS = {
    "full": "Fully shared FT-Transformer (post-hoc P(O1) x P(O2|X,death))",
}

print("Configuration loaded.")


## 3. Load the MI dataset

In [ ]:
# df column index j (0-indexed, header=None) holds UCI attribute (j+1) --
# e.g. column 1 = attribute 2 = AGE.
UCI_COL_NAMES = {
    1: "AGE", 2: "SEX", 3: "INF_ANAM", 4: "STENOK_AN", 5: "FK_STENOK", 6: "IBS_POST",
    7: "IBS_NASL", 8: "GB", 9: "SIM_GIPERT", 10: "DLIT_AG", 11: "ZSN_A", 12: "nr11",
    13: "nr01", 14: "nr02", 15: "nr03", 16: "nr04", 17: "nr07", 18: "nr08",
    19: "np01", 20: "np04", 21: "np05", 22: "np07", 23: "np08", 24: "np09",
    25: "np10", 26: "endocr_01", 27: "endocr_02", 28: "endocr_03", 29: "zab_leg_01", 30: "zab_leg_02",
    31: "zab_leg_03", 32: "zab_leg_04", 33: "zab_leg_06", 34: "S_AD_KBRIG", 35: "D_AD_KBRIG", 36: "S_AD_ORIT",
    37: "D_AD_ORIT", 38: "O_L_POST", 39: "K_SH_POST", 40: "MP_TP_POST", 41: "SVT_POST", 42: "GT_POST",
    43: "FIB_G_POST", 44: "ant_im", 45: "lat_im", 46: "inf_im", 47: "post_im", 48: "IM_PG_P",
    49: "ritm_ecg_p_01", 50: "ritm_ecg_p_02", 51: "ritm_ecg_p_04", 52: "ritm_ecg_p_06", 53: "ritm_ecg_p_07", 54: "ritm_ecg_p_08",
    55: "n_r_ecg_p_01", 56: "n_r_ecg_p_02", 57: "n_r_ecg_p_03", 58: "n_r_ecg_p_04", 59: "n_r_ecg_p_05", 60: "n_r_ecg_p_06",
    61: "n_r_ecg_p_08", 62: "n_r_ecg_p_09", 63: "n_r_ecg_p_10", 64: "n_p_ecg_p_01", 65: "n_p_ecg_p_03", 66: "n_p_ecg_p_04",
    67: "n_p_ecg_p_05", 68: "n_p_ecg_p_06", 69: "n_p_ecg_p_07", 70: "n_p_ecg_p_08", 71: "n_p_ecg_p_09", 72: "n_p_ecg_p_10",
    73: "n_p_ecg_p_11", 74: "n_p_ecg_p_12", 75: "fibr_ter_01", 76: "fibr_ter_02", 77: "fibr_ter_03", 78: "fibr_ter_05",
    79: "fibr_ter_06", 80: "fibr_ter_07", 81: "fibr_ter_08", 82: "GIPO_K", 83: "K_BLOOD", 84: "GIPER_Na",
    85: "Na_BLOOD", 86: "ALT_BLOOD", 87: "AST_BLOOD", 88: "KFK_BLOOD", 89: "L_BLOOD", 90: "ROE",
    91: "TIME_B_S", 92: "R_AB_1_n", 93: "R_AB_2_n", 94: "R_AB_3_n", 95: "NA_KB", 96: "NOT_NA_KB",
    97: "LID_KB", 98: "NITR_S", 99: "NA_R_1_n", 100: "NA_R_2_n", 101: "NA_R_3_n", 102: "NOT_NA_1_n",
    103: "NOT_NA_2_n", 104: "NOT_NA_3_n", 105: "LID_S_n", 106: "B_BLOK_S_n", 107: "ANT_CA_S_n", 108: "GEPAR_S_n",
    109: "ASP_S_n", 110: "TIKL_S_n", 111: "TRENT_S_n",
}

def load_dataset_raw(path, excl_attrs=None):
    """excl_attrs: set of UCI attribute numbers (1-indexed, per the dataset's own
    documentation) to exclude as not-yet-available at the chosen prediction time point.
    Defaults to the admission-time exclusion set. See the time-window section below for
    the 24h/48h/72h alternatives used in Experiment 1 (time-window ablation)."""
    df = pd.read_csv(path, header=None)
    df = df.replace("?", np.nan).apply(pd.to_numeric, errors="coerce")

    # Admission-time-only feature set: exclude attributes 93-95 (R_AB_1_n/2_n/3_n, pain
    # relapse) and 100-105 (NA_R_1_n/2_n/3_n, NOT_NA_1_n/2_n/3_n -- opioids/NSAIDs given
    # during the ICU stay), since none of those are known at the time of admission.
    if excl_attrs is None:
        excl_attrs = {93, 94, 95, 100, 101, 102, 103, 104, 105}
    EXCL = {0} | {a - 1 for a in excl_attrs}
    feat_idx = [i for i in range(1, 112) if i not in EXCL]

    fc = df.iloc[:, feat_idx].copy()
    fc.columns = [UCI_COL_NAMES[i] for i in feat_idx]

    X_raw = fc.values.astype(np.float64)
    y_comp = df.iloc[:, 112:123].values.astype(np.float32)
    y_mort = df.iloc[:, 123].values.astype(np.int64)
    y_bin = (y_mort > 0).astype(np.int64)

    print("Raw shape:", df.shape)
    print("Admission features retained:", X_raw.shape[1])
    print("Patients:", len(X_raw))
    print("Alive / died:", int((y_bin == 0).sum()), "/", int((y_bin == 1).sum()))
    print("Cause distribution:", dict(zip(*np.unique(y_mort, return_counts=True))))
    return X_raw, y_comp, y_mort, y_bin, fc.columns.tolist()

X_raw, y_comp, y_mort, y_binary, FEATURE_NAMES = load_dataset_raw(DATASET_PATH)
P["n_features"] = X_raw.shape[1]
assert X_raw.shape[1] == 102, f"expected all 102 admission-time candidates, got {X_raw.shape[1]}"
print("First 10 feature names:", FEATURE_NAMES[:10])


## 4. Time-window feature sets

The UCI dataset defines four cumulative prediction time points, each adding variables that become available as the hospital stay progresses: **admission**, **+24h**, **+48h**, **+72h** (full stay). Used by the time-window ablation (Experiment 1) and the windowed SHAP analysis (Experiment 3).

In [ ]:
TIME_WINDOWS = ["admission", "24h", "48h", "72h"]

WINDOW_EXCL_ATTRS = {
    "admission": {93, 94, 95, 100, 101, 102, 103, 104, 105},
    "24h":       {94, 95, 101, 102, 104, 105},
    "48h":       {95, 102, 105},
    "72h":       set(),
}

WINDOW_LABELS = {
    "admission": "Admission only",
    "24h": "Admission + 24h",
    "48h": "Admission + 24h + 48h",
    "72h": "Admission + 24h + 48h + 72h (full stay)",
}

def load_window(window):
    """Loads the feature matrix for one time window. Labels are asserted identical
    to the already-loaded admission-window labels (y_comp/y_mort/y_binary) -- only the
    INPUT features differ between windows, never the outcomes being predicted."""
    Xw, ycw, ymw, ybw, featw = load_dataset_raw(DATASET_PATH, WINDOW_EXCL_ATTRS[window])
    assert np.array_equal(ycw, y_comp), "y_comp changed across windows -- should be impossible"
    assert np.array_equal(ymw, y_mort), "y_mort changed across windows -- should be impossible"
    assert np.array_equal(ybw, y_binary), "y_binary changed across windows -- should be impossible"
    return Xw, featw

for w in TIME_WINDOWS:
    n = len(WINDOW_EXCL_ATTRS[w])
    print(f"  {w:10s} ({WINDOW_LABELS[w]}): excludes {n} attribute(s) -> "
          f"{111 - n} candidate features")


## 5. Global fold assignment (multilabel-stratified, shared across every experiment)

In [ ]:
# ================================================================
# Global, multilabel-stratified fold assignment (shared by every model below)
# ================================================================
N_FOLDS_REQUESTED = 5

def choose_n_folds(y_mort, n_folds_requested):
    """Reduce fold count if the smallest cause-of-death class can't appear in every fold."""
    counts = np.bincount(y_mort[y_mort > 0], minlength=8)[1:]
    counts = counts[counts > 0]
    smallest = counts.min() if len(counts) else n_folds_requested
    n_folds = int(min(n_folds_requested, max(2, smallest)))
    if n_folds < n_folds_requested:
        print(f"Smallest cause-of-death class has {smallest} patients; "
              f"reducing folds from {n_folds_requested} to {n_folds} so every "
              f"cause appears in every fold.")
    return n_folds

def build_stratification_labels(y_mort, y_comp):
    mort_onehot = np.eye(8)[y_mort]  # 8 mortality classes: 0=alive, 1..7=cause
    return np.concatenate([mort_onehot, y_comp], axis=1).astype(int)

print("Cause-of-death class counts:", dict(zip(*np.unique(y_mort, return_counts=True))))
N_FOLDS = choose_n_folds(y_mort, N_FOLDS_REQUESTED)
print("Using", N_FOLDS, "outer folds.")

strat_labels = build_stratification_labels(y_mort, y_comp)
splitter = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

GLOBAL_FOLD_ID = np.full(len(X_raw), -1, dtype=int)
for fold_i, (_, test_idx) in enumerate(splitter.split(X_raw, strat_labels)):
    GLOBAL_FOLD_ID[test_idx] = fold_i
assert (GLOBAL_FOLD_ID >= 0).all(), "every patient must be assigned to a fold"

def iter_fold_splits(fold_ids, n_folds):
    """Yield (train_idx, val_idx), same contract as sklearn's KFold.split, from a
    precomputed fold assignment so every model uses IDENTICAL folds."""
    idx_all = np.arange(len(fold_ids))
    for f in range(n_folds):
        val = idx_all[fold_ids == f]
        tr = idx_all[fold_ids != f]
        yield tr, val

print("Patients per fold:", pd.Series(GLOBAL_FOLD_ID).value_counts().sort_index().to_dict())


## 6. Fold-safe preprocessing and dataset

In [ ]:
class FoldImputer:
    def __init__(self):
        self.medians_ = None

    def fit_transform(self, X):
        X = X.copy()
        self.medians_ = np.nanmedian(X, axis=0)
        mask = np.isnan(X)
        X[mask] = np.take(self.medians_, np.where(mask)[1])
        return X.astype(np.float32)

    def transform(self, X):
        X = X.copy()
        mask = np.isnan(X)
        X[mask] = np.take(self.medians_, np.where(mask)[1])
        return X.astype(np.float32)

def make_preprocessor(X_train_raw):
    imp = FoldImputer()
    X_imp = imp.fit_transform(X_train_raw)
    sc = StandardScaler()
    X_scaled = sc.fit_transform(X_imp).astype(np.float32)
    return imp, sc, X_scaled

def apply_preprocessor(imp, sc, X_raw):
    return sc.transform(imp.transform(X_raw)).astype(np.float32)

class MIDataset(Dataset):
    def __init__(self, X, y_comp, y_mort, y_bin):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_comp = torch.tensor(y_comp, dtype=torch.float32)
        self.y_mort = torch.tensor(y_mort, dtype=torch.long)
        self.y_bin = torch.tensor(y_bin, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y_comp[i], self.y_mort[i], self.y_bin[i]


## 7. Shared FT-Transformer building blocks

In [ ]:
class FeatureTokenizer(nn.Module):
    def __init__(self, n, d):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n, d))
        self.b = nn.Parameter(torch.zeros(n, d))
        self.cls = nn.Parameter(torch.empty(1, 1, d))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))
        nn.init.normal_(self.cls, std=0.02)

    def forward(self, x):
        t = x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)
        return torch.cat([self.cls.expand(x.size(0), -1, -1), t], dim=1)

class TransformerBlock(nn.Module):
    def __init__(self, d, nh, df, ad, fd, rd):
        super().__init__()
        self.n1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, nh, dropout=ad, batch_first=True)
        self.n2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(
            nn.Linear(d, df), nn.GELU(), nn.Dropout(fd), nn.Linear(df, d)
        )
        self.drop = nn.Dropout(rd)

    def forward(self, x):
        n = self.n1(x)
        a, w = self.attn(n, n, n, need_weights=True, average_attn_weights=False)
        x = x + self.drop(a)
        return x + self.drop(self.ffn(self.n2(x))), w

class TaskHead(nn.Module):
    def __init__(self, d, n, drop):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d), nn.Dropout(drop),
            nn.Linear(d, d // 2), nn.GELU(),
            nn.Dropout(drop / 2), nn.Linear(d // 2, n)
        )

    def forward(self, x): return self.net(x)

def make_encoder(p):
    """One fresh tokenizer + stack of transformer blocks (one encoder instance)."""
    d = p["d_token"]
    df = d * p["d_ffn_factor"]
    tok = FeatureTokenizer(p["n_features"], d)
    blocks = nn.ModuleList([
        TransformerBlock(d, p["n_heads"], df, p["attn_dropout"], p["ffn_dropout"], p["res_dropout"])
        for _ in range(p["n_layers"])
    ])
    return tok, blocks

def run_encoder(tok, blocks, x):
    t = tok(x)
    for blk in blocks:
        t, _ = blk(t)
    return t[:, 0, :]  # CLS representation


In [ ]:
def cls_attention_importance(tok, blocks, x):
    """CLS-to-feature attention magnitude for one forward pass, averaged over heads
    and over transformer blocks (there is only one block with the current config).
    Returns a (batch, n_features) numpy array."""
    t = tok(x)
    attn_last = None
    for blk in blocks:
        t, w = blk(t)  # w: (B, n_heads, seq_len, seq_len)
        attn_last = w
    cls_to_feat = attn_last[:, :, 0, 1:]       # attention FROM the CLS token TO each feature token
    cls_to_feat = cls_to_feat.mean(dim=1)      # average over heads -> (B, n_features)
    return cls_to_feat.detach().cpu().numpy()


## 8. Multi-task FT-Transformer (single shared encoder, three task heads)

In [ ]:
class FTTransformerSharing(nn.Module):
    """
    O1 = P(death | X)                       (binary)
    O2 = P(cause | death, X)                 (7-way, trained on deceased patients only)
    O3 = P(complication | X)                 (11-way multi-label, independent probabilities)

    The paper uses a single fully-shared encoder ("full"): one FT-Transformer backbone
    produces one [CLS] representation that feeds all three task heads, and O2 is combined
    with O1 post-hoc as P(death | X) x P(cause | death, X) (Eq. 4). `sharing` is kept as a
    constructor argument (rather than hard-coded) purely so the model class stays reusable,
    but only sharing="full" is ever instantiated in this notebook.
    """
    VALID_SHARING = ("full",)

    def __init__(self, p, sharing="full"):
        super().__init__()
        if sharing not in self.VALID_SHARING:
            raise ValueError(f"sharing must be one of {self.VALID_SHARING}, got {sharing!r}")
        self.sharing = sharing
        d = p["d_token"]

        self.tok_shared, self.enc_shared = make_encoder(p)

        dr = p["head_dropout"]
        self.o1_head = TaskHead(d, 1, dr)
        self.o2_head = TaskHead(d, p["n_causes"], dr)
        self.o3_head = TaskHead(d, p["n_complications"], dr)

    def forward(self, x):
        cls_shared = run_encoder(self.tok_shared, self.enc_shared, x)
        cls_o1 = cls_o2 = cls_o3 = cls_shared

        o1 = self.o1_head(cls_o1)  # (B, 1) mortality logit
        o2 = self.o2_head(cls_o2)  # (B, n_causes) cause logits
        o3 = self.o3_head(cls_o3)  # (B, n_complications) complication logits
        return o1, o2, o3

    def n_encoder_params(self):
        """Convenience: count parameters that live in the encoder vs. the heads."""
        head_params = sum(p_.numel() for p_ in list(self.o1_head.parameters()) +
                           list(self.o2_head.parameters()) + list(self.o3_head.parameters()))
        total_params = sum(p_.numel() for p_ in self.parameters())
        return {"encoder_params": total_params - head_params, "head_params": head_params, "total_params": total_params}


## 9. Multi-task O1/O2/O3 loss

In [ ]:
class MultiTaskLoss(nn.Module):
    """Weighted sum of O1 (mortality BCE), O2 (cause CE on deceased only), O3 (complication BCE)."""
    def __init__(self, p, comp_pw, cause_cw, mort_pos_weight):
        super().__init__()
        self.register_buffer("cause_cw", cause_cw)
        self.register_buffer("comp_pw", comp_pw)
        self.register_buffer("mort_pw", torch.tensor(float(mort_pos_weight)))
        self.w_mort = p["w_mort"]
        self.w_cause = p["w_cause"]
        self.w_comp = p["w_comp"]

    def forward(self, o1, o2, o3, y_bin, y_mort, y_comp):
        l_m = F.binary_cross_entropy_with_logits(o1.squeeze(-1), y_bin, pos_weight=self.mort_pw)

        dead = y_bin > 0.5
        if dead.sum() > 0:
            cause_target = (y_mort[dead] - 1).clamp(min=0)
            l_c = F.cross_entropy(o2[dead], cause_target, weight=self.cause_cw)
        else:
            l_c = torch.tensor(0.0, device=o1.device)

        l_o = F.binary_cross_entropy_with_logits(o3, y_comp, pos_weight=self.comp_pw)

        total = self.w_mort * l_m + self.w_cause * l_c + self.w_comp * l_o
        return total, l_m.item(), l_c.item(), l_o.item()

def make_loss(p, y_comp_tr, y_mort_tr, y_bin_tr, device):
    n_pos = y_comp_tr.sum(0).clip(min=1)
    n_neg = len(y_comp_tr) - n_pos
    comp_pw = torch.tensor(
        np.minimum(n_neg / n_pos, p["comp_cap"]), dtype=torch.float32, device=device
    )

    dead = y_bin_tr == 1
    y_c = (y_mort_tr[dead] - 1).clip(min=0)
    counts = np.bincount(y_c, minlength=p["n_causes"]).clip(min=1).astype(float)
    cause_w = np.minimum(
        counts.sum() / (p["n_causes"] * counts), p["cause_weight_cap"]
    )
    cause_cw = torch.tensor(cause_w, dtype=torch.float32, device=device)

    return MultiTaskLoss(p, comp_pw, cause_cw, p["mort_pos_weight"]).to(device)

class EarlyStopping:
    def __init__(self, patience, delta=1e-4):
        self.patience = patience
        self.delta = delta
        self.best = float("inf")
        self.bad = 0
        self.state = None

    def step(self, value, model):
        if value < self.best - self.delta:
            self.best = value
            self.bad = 0
            self.state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            self.bad += 1
        return self.bad >= self.patience

    def restore(self, model):
        if self.state is not None:
            model.load_state_dict(self.state)


## 10. Training / evaluation helpers, and the hierarchical O1 x O2 coupling (Eq. 4)

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total = 0.0
    n = 0
    for Xb, yc, ym, yb in loader:
        Xb, yc = Xb.to(DEVICE), yc.to(DEVICE)
        ym, yb = ym.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        o1, o2, o3 = model(Xb)
        loss, _, _, _ = loss_fn(o1, o2, o3, yb, ym, yc)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * Xb.size(0)
        n += Xb.size(0)
    return total / max(n, 1)

@torch.no_grad()
def predict(model, loader, loss_fn):
    model.eval()
    o1_all, o2_all, o3_all, yb_all, ym_all, yc_all = [], [], [], [], [], []
    total = 0.0
    n = 0
    for Xb, yc, ym, yb in loader:
        Xb, yc = Xb.to(DEVICE), yc.to(DEVICE)
        ym, yb = ym.to(DEVICE), yb.to(DEVICE)
        o1, o2, o3 = model(Xb)
        loss, _, _, _ = loss_fn(o1, o2, o3, yb, ym, yc)
        total += loss.item() * Xb.size(0)
        n += Xb.size(0)
        o1_all.append(torch.sigmoid(o1).cpu())
        o2_all.append(o2.cpu())
        o3_all.append(torch.sigmoid(o3).cpu())
        yb_all.append(yb.cpu())
        ym_all.append(ym.cpu())
        yc_all.append(yc.cpu())

    return (
        total / max(n, 1),
        torch.cat(o1_all).numpy().reshape(-1),
        torch.cat(o2_all).numpy(),
        torch.cat(o3_all).numpy(),
        torch.cat(yb_all).numpy().astype(int),
        torch.cat(ym_all).numpy().astype(int),
        torch.cat(yc_all).numpy().astype(int),
    )

def stable_softmax(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(axis=1, keepdims=True)

def hierarchical_probabilities(p_dead, o2_logits):
    """
    Returns probabilities for the mutually exclusive 8-way outcome:
      column 0 = alive
      columns 1..7 = death through cause k

    Computed for EVERY patient; no O1 threshold is used (matches Eq. 4 in the paper).
    """
    p_dead = np.clip(np.asarray(p_dead), 1e-7, 1 - 1e-7)
    p_cause_given_dead = stable_softmax(o2_logits)
    p_alive = 1.0 - p_dead
    p_fatal_cause = p_dead[:, None] * p_cause_given_dead
    p8 = np.column_stack([p_alive, p_fatal_cause])
    return p8, p_cause_given_dead, p_fatal_cause

def o1_metrics(y_true, p_dead):
    auc = roc_auc_score(y_true, p_dead)
    pred = (p_dead >= 0.5).astype(int)
    return {
        "auc": auc,
        "f1_at_0.5": f1_score(y_true, pred, zero_division=0),
        "precision_at_0.5": precision_score(y_true, pred, zero_division=0),
        "recall_at_0.5": recall_score(y_true, pred, zero_division=0),
        "brier": brier_score_loss(y_true, p_dead),
    }

def o3_metrics(y_comp, p_comp):
    """Macro AUC / F1 across the 11 (independent, multi-label) complications."""
    n_comp = y_comp.shape[1]
    aucs = []
    for j in range(n_comp):
        yj = y_comp[:, j]
        if yj.sum() and (1 - yj).sum():
            aucs.append(roc_auc_score(yj, p_comp[:, j]))
    macro_auc = float(np.mean(aucs)) if aucs else np.nan
    pred = (p_comp >= 0.5).astype(int)
    macro_f1 = f1_score(y_comp, pred, average="macro", zero_division=0)
    return {"o3_macro_auc": macro_auc, "o3_macro_f1": macro_f1}

def hierarchical_metrics(p8, y_mort):
    """Metrics on all patients without an O1 binary gate."""
    pred8 = p8.argmax(axis=1)
    true8 = y_mort.copy()

    acc = accuracy_score(true8, pred8)
    f1 = f1_score(true8, pred8, average="macro", zero_division=0)

    per_cause_auc = {}
    for k in range(1, 8):
        yk = (true8 == k).astype(int)
        if yk.sum() and (1 - yk).sum():
            per_cause_auc[k] = roc_auc_score(yk, p8[:, k])
        else:
            per_cause_auc[k] = np.nan

    mean_auc = float(np.nanmean(list(per_cause_auc.values())))
    ll = log_loss(true8, np.clip(p8, 1e-8, 1 - 1e-8), labels=np.arange(8))

    dead = true8 > 0
    cause_correct_among_dead = ((pred8 == true8) & dead).sum()
    system_recall = cause_correct_among_dead / max(dead.sum(), 1)

    if dead.sum():
        cond_true = true8[dead] - 1
        cond_prob = p8[dead, 1:]
        cond_pred = cond_prob.argmax(axis=1)
        cond_acc = accuracy_score(cond_true, cond_pred)
        cond_f1 = f1_score(cond_true, cond_pred, average="macro", zero_division=0)
        cond_auc_list = []
        for k in range(7):
            yk = (cond_true == k).astype(int)
            if yk.sum() and (1 - yk).sum():
                cond_auc_list.append(roc_auc_score(yk, cond_prob[:, k]))
        cond_auc = float(np.mean(cond_auc_list)) if cond_auc_list else np.nan
    else:
        cond_acc = cond_f1 = cond_auc = np.nan

    return {
        "end_to_end_acc": acc,
        "end_to_end_macro_f1": f1,
        "end_to_end_mean_fatal_cause_auc": mean_auc,
        "end_to_end_log_loss": ll,
        "system_recall_among_deaths": system_recall,
        "o2_conditional_acc_dead": cond_acc,
        "o2_conditional_macro_f1_dead": cond_f1,
        "o2_conditional_mean_auc_dead": cond_auc,
        "pred8": pred8,
        "per_cause_auc": per_cause_auc,
    }


## 11. Cross-validation runner

In [ ]:
def run_cv(sharing, X_raw, y_comp, y_mort, y_bin, p, fold_ids, n_folds, verbose=True):
    fold_rows = []
    oof = {
        "p_dead": np.zeros(len(X_raw)),
        "p_cause_given_dead": np.zeros((len(X_raw), p["n_causes"])),
        "p_fatal_cause": np.zeros((len(X_raw), p["n_causes"])),
        "p_comp": np.zeros((len(X_raw), p["n_complications"])),
        "p8": np.zeros((len(X_raw), 8)),
        "o2_logit": np.zeros((len(X_raw), p["n_causes"])),
        "fold": np.full(len(X_raw), -1, dtype=int),
    }
    attn_sum = np.zeros(p["n_features"])   # accumulated only when sharing == "full"
    attn_n = 0

    for fold, (tr, vl) in enumerate(iter_fold_splits(fold_ids, n_folds), 1):
        if verbose:
            print("\n" + "=" * 72)
            print(f"[{sharing}] FOLD {fold}/{n_folds}")

        imp, sc, Xtr = make_preprocessor(X_raw[tr])
        Xvl = apply_preprocessor(imp, sc, X_raw[vl])

        tr_dl = DataLoader(
            MIDataset(Xtr, y_comp[tr], y_mort[tr], y_bin[tr]),
            batch_size=p["batch_size"], shuffle=True, drop_last=False
        )
        vl_dl = DataLoader(
            MIDataset(Xvl, y_comp[vl], y_mort[vl], y_bin[vl]),
            batch_size=p["batch_size"], shuffle=False
        )

        loss_fn = make_loss(p, y_comp[tr], y_mort[tr], y_bin[tr], DEVICE)
        model = FTTransformerSharing(p, sharing=sharing).to(DEVICE)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=p["lr"], weight_decay=p["weight_decay"]
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=p["epochs"]
        )
        stopper = EarlyStopping(p["patience"])

        for epoch in range(1, p["epochs"] + 1):
            train_loss = train_epoch(model, tr_dl, optimizer, loss_fn)
            scheduler.step()
            val_loss, *_ = predict(model, vl_dl, loss_fn)

            if verbose and epoch % 30 == 0:
                print(f"  epoch {epoch:3d}: train={train_loss:.4f}, val={val_loss:.4f}")
            if stopper.step(val_loss, model):
                if verbose:
                    print(f"  early stop at epoch {epoch}")
                break

        stopper.restore(model)

        # Held-out predictions. O2 is retained for ALL validation patients.
        _, p_dead_vl, o2lg_vl, p_comp_vl, yb_vl, ym_vl, yc_vl = predict(model, vl_dl, loss_fn)
        p8_vl, p_cause_vl, p_fatal_vl = hierarchical_probabilities(p_dead_vl, o2lg_vl)

        m1 = o1_metrics(yb_vl, p_dead_vl)
        m3 = o3_metrics(yc_vl, p_comp_vl)
        mh = hierarchical_metrics(p8_vl, ym_vl)

        if verbose:
            print(f"  O1 AUC={m1['auc']:.4f} | Brier={m1['brier']:.4f}")
            print(f"  O3 macro AUC={m3['o3_macro_auc']:.4f} | macro F1={m3['o3_macro_f1']:.4f}")
            print(f"  O2 conditional AUC (dead only)={mh['o2_conditional_mean_auc_dead']:.4f}")
            print(f"  Fatal-cause AUC (all patients)={mh['end_to_end_mean_fatal_cause_auc']:.4f}")
            print(f"  End-to-end macro-F1={mh['end_to_end_macro_f1']:.4f}")
            print(f"  System recall among deaths={mh['system_recall_among_deaths']:.4f}")

        fold_rows.append({
            "sharing": sharing,
            "fold": fold,
            "o1_auc": m1["auc"],
            "o1_brier": m1["brier"],
            "o1_f1_at_0.5": m1["f1_at_0.5"],
            "o3_macro_auc": m3["o3_macro_auc"],
            "o3_macro_f1": m3["o3_macro_f1"],
            "o2_conditional_auc_dead": mh["o2_conditional_mean_auc_dead"],
            "o2_conditional_acc_dead": mh["o2_conditional_acc_dead"],
            "o2_conditional_f1_dead": mh["o2_conditional_macro_f1_dead"],
            "fatal_cause_auc_all": mh["end_to_end_mean_fatal_cause_auc"],
            "end_to_end_acc": mh["end_to_end_acc"],
            "end_to_end_macro_f1": mh["end_to_end_macro_f1"],
            "end_to_end_log_loss": mh["end_to_end_log_loss"],
            "system_recall": mh["system_recall_among_deaths"],
        })

        oof["p_dead"][vl] = p_dead_vl
        oof["p_cause_given_dead"][vl] = p_cause_vl
        oof["p_fatal_cause"][vl] = p_fatal_vl
        oof["p_comp"][vl] = p_comp_vl
        oof["p8"][vl] = p8_vl
        oof["o2_logit"][vl] = o2lg_vl
        oof["fold"][vl] = fold - 1

        # Attention-based feature importance: meaningful for the fully-shared encoder,
        # where one representation feeds all three heads.
        if sharing == "full":
            model.eval()
            with torch.no_grad():
                Xvl_t = torch.tensor(Xvl, dtype=torch.float32).to(DEVICE)
                attn_fold = cls_attention_importance(model.tok_shared, model.enc_shared, Xvl_t)
            attn_sum += attn_fold.sum(axis=0)
            attn_n += attn_fold.shape[0]

    if sharing == "full" and attn_n > 0:
        oof["attention_importance"] = attn_sum / attn_n

    return pd.DataFrame(fold_rows), oof, model.n_encoder_params()


## 12. Run 5-fold CV for the fully-shared FT-Transformer (admission-time features)

This is the primary model reported throughout the paper. Results feed directly into Table 2, Table 4 (admission row), Table 5, and Appendix Table 10.

In [ ]:
all_results = {}
all_oof = {}
param_counts = {}

for sharing in CONDITIONS:
    print("\n" + "#" * 72)
    print(f"# CONDITION: {sharing} \u2014 {CONDITION_LABELS[sharing]}")
    print("#" * 72)
    results_df, oof, n_params = run_cv(
        sharing, X_raw, y_comp, y_mort, y_binary, P,
        fold_ids=GLOBAL_FOLD_ID, n_folds=N_FOLDS, verbose=True,
    )
    all_results[sharing] = results_df
    all_oof[sharing] = oof
    param_counts[sharing] = n_params
    results_df.to_csv(f"{SAVE_DIR}/cv_results_{sharing}.csv", index=False)

    print(f"\n[{sharing}] {N_FOLDS}-fold CV summary (fold-level results -- Appendix Table 10 style)")
    display(results_df.round(4))
    print(results_df.mean(numeric_only=True))


# Section 4.1: Comparison with Prior Work -- Parallel Classical Pipeline

How well do standard ML models perform on the same admission-only features and the same global folds as the FT-Transformer? Six algorithms (Logistic Regression, Random Forest, Extra Trees, Gradient Boosting, XGBoost, LightGBM) are compared for O1 and O2; Random Forest / Extra Trees multi-output wrappers are compared for O3. No SMOTE is used anywhere -- class imbalance is handled with `sample_weight='balanced'` (classical models) or positive-class weighting (FT-Transformer), matching the paper's Section 3.3.

In [ ]:
# ================================================================
# Classical-model imports for the Parallel Pipeline baseline
# ================================================================
!pip -q install -U xgboost lightgbm

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import lightgbm as lgb

def select_threshold_f1(p, y):
    """Sweep thresholds and return the one maximizing F1 on the given (usually
    training-fold) predictions -- matches the paper's 'F1-optimal on the training
    fold' threshold-selection rule for classical O1 models."""
    best_thr, best_f1 = 0.5, -1.0
    for thr in np.linspace(0.01, 0.99, 99):
        pred = (p >= thr).astype(int)
        f1 = f1_score(y, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return float(best_thr)

def make_classical_models(task):
    """task in {'binary', 'multiclass'}. Every model handles class imbalance through
    sample_weight='balanced' -- no SMOTE anywhere, matching the paper's Section 3.3."""
    xgb_metric = "logloss" if task == "binary" else "mlogloss"
    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=SEED),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
        "GradientBoosting": GradientBoostingClassifier(random_state=SEED),
        "XGBoost": xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                      eval_metric=xgb_metric, random_state=SEED, n_jobs=-1, verbosity=0),
        "LightGBM": lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=SEED, verbosity=-1),
    }


## 13.1 O1 leaderboard (binary mortality)

In [ ]:
def run_parallel_pipeline_o1(X_raw, y_bin, fold_ids, n_folds, verbose=True):
    models = make_classical_models("binary")
    rows = []
    oof_proba = {name: np.zeros(len(X_raw)) for name in models}
    fold_importance = {name: [] for name in models}

    for fold, (tr, vl) in enumerate(iter_fold_splits(fold_ids, n_folds), 1):
        imp, sc, Xtr = make_preprocessor(X_raw[tr])
        Xvl = apply_preprocessor(imp, sc, X_raw[vl])
        ytr, yvl = y_bin[tr], y_bin[vl]
        sw = compute_sample_weight("balanced", ytr)

        for name, base_model in models.items():
            model = clone(base_model)
            model.fit(Xtr, ytr, sample_weight=sw)
            idx1 = list(model.classes_).index(1)
            proba_tr = model.predict_proba(Xtr)[:, idx1]
            proba_vl = model.predict_proba(Xvl)[:, idx1]
            thr = select_threshold_f1(proba_tr, ytr)
            pred_vl = (proba_vl >= thr).astype(int)

            auc = roc_auc_score(yvl, proba_vl) if yvl.sum() and (1 - yvl).sum() else np.nan
            f1 = f1_score(yvl, pred_vl, zero_division=0)
            prec = precision_score(yvl, pred_vl, zero_division=0)
            rec = recall_score(yvl, pred_vl, zero_division=0)
            tn = int(((pred_vl == 0) & (yvl == 0)).sum())
            fp = int(((pred_vl == 1) & (yvl == 0)).sum())
            spec = tn / max(tn + fp, 1)
            brier = brier_score_loss(yvl, proba_vl)

            rows.append({
                "algorithm": name, "fold": fold, "auc": auc, "f1": f1, "precision": prec,
                "recall": rec, "specificity": spec, "brier": brier, "threshold": thr,
            })
            oof_proba[name][vl] = proba_vl
            if hasattr(model, "feature_importances_"):
                fold_importance[name].append(model.feature_importances_)

        if verbose:
            print(f"[Parallel Pipeline O1] fold {fold}/{n_folds} done")

    rows_df = pd.DataFrame(rows)
    leaderboard = (
        rows_df.groupby("algorithm").mean(numeric_only=True)
        .drop(columns=["fold"]).sort_values("auc", ascending=False)
    )
    return rows_df, leaderboard, oof_proba, fold_importance

o1_pp_rows, o1_pp_leaderboard, o1_pp_oof_proba, o1_pp_importance = run_parallel_pipeline_o1(
    X_raw, y_binary, GLOBAL_FOLD_ID, N_FOLDS, verbose=True
)
o1_pp_rows.to_csv(f"{SAVE_DIR}/parallel_pipeline_o1_per_fold.csv", index=False)
o1_pp_leaderboard.to_csv(f"{SAVE_DIR}/parallel_pipeline_o1_leaderboard.csv")
best_o1_name = o1_pp_leaderboard.index[0]
oof_parallel_p_dead = o1_pp_oof_proba[best_o1_name]
print("\nO1 leaderboard (mean across folds, ranked by AUC):")
display(o1_pp_leaderboard.round(4))
print("Best O1 classical model:", best_o1_name)


## 13.2 O2 leaderboard (cause of death, deceased sub-cohort)

In [ ]:
def run_parallel_pipeline_o2(X_raw, y_comp, y_mort, fold_ids, n_folds, verbose=True):
    models = make_classical_models("multiclass")
    n_causes = P["n_causes"]
    rows = []
    oof_proba = {name: np.zeros((len(X_raw), n_causes)) for name in models}

    for fold, (tr, vl) in enumerate(iter_fold_splits(fold_ids, n_folds), 1):
        imp, sc, Xtr_all = make_preprocessor(X_raw[tr])
        Xvl_all = apply_preprocessor(imp, sc, X_raw[vl])

        dead_tr = y_mort[tr] > 0
        Xtr = Xtr_all[dead_tr]
        ytr = y_mort[tr][dead_tr] - 1  # 0..6, matches the FTT's masked cause target
        dead_vl_mask = y_mort[vl] > 0
        yvl_dead = y_mort[vl][dead_vl_mask] - 1

        if dead_tr.sum() < 2 or len(np.unique(ytr)) < 2:
            if verbose:
                print(f"[Parallel Pipeline O2] fold {fold}: too few deceased training patients, skipping")
            continue
        sw = compute_sample_weight("balanced", ytr)

        for name, base_model in models.items():
            model = clone(base_model)
            try:
                model.fit(Xtr, ytr, sample_weight=sw)
            except Exception as e:
                if verbose:
                    print(f"[Parallel Pipeline O2] {name} fold {fold} failed: {e}")
                continue

            proba_vl_all = model.predict_proba(Xvl_all)  # predicted for every val patient
            aligned = np.zeros((Xvl_all.shape[0], n_causes))
            for j, c in enumerate(model.classes_):
                aligned[:, int(c)] = proba_vl_all[:, j]
            oof_proba[name][vl] = aligned

            if dead_vl_mask.sum() == 0:
                continue
            proba_dead = aligned[dead_vl_mask]
            pred_dead = proba_dead.argmax(axis=1)

            auc_list = []
            for k in range(n_causes):
                yk = (yvl_dead == k).astype(int)
                if yk.sum() and (1 - yk).sum():
                    auc_list.append(roc_auc_score(yk, proba_dead[:, k]))
            ovr_auc = float(np.mean(auc_list)) if auc_list else np.nan
            macro_f1 = f1_score(yvl_dead, pred_dead, average="macro", zero_division=0)
            acc = accuracy_score(yvl_dead, pred_dead)

            rows.append({"algorithm": name, "fold": fold, "ovr_auc": ovr_auc,
                         "macro_f1": macro_f1, "accuracy": acc})

        if verbose:
            print(f"[Parallel Pipeline O2] fold {fold}/{n_folds} done ({int(dead_vl_mask.sum())} deceased in val)")

    rows_df = pd.DataFrame(rows)
    leaderboard = (
        rows_df.groupby("algorithm").mean(numeric_only=True)
        .drop(columns=["fold"]).sort_values("ovr_auc", ascending=False)
    )
    return rows_df, leaderboard, oof_proba

o2_pp_rows, o2_pp_leaderboard, o2_pp_oof_proba = run_parallel_pipeline_o2(
    X_raw, y_comp, y_mort, GLOBAL_FOLD_ID, N_FOLDS, verbose=True
)
o2_pp_rows.to_csv(f"{SAVE_DIR}/parallel_pipeline_o2_per_fold.csv", index=False)
o2_pp_leaderboard.to_csv(f"{SAVE_DIR}/parallel_pipeline_o2_leaderboard.csv")
best_o2_name = o2_pp_leaderboard.index[0]
oof_parallel_p_cause = o2_pp_oof_proba[best_o2_name]
print("\nO2 leaderboard (deceased sub-cohort, mean across folds, ranked by OvR AUC):")
display(o2_pp_leaderboard.round(4))
print("Best O2 classical model:", best_o2_name)


## 13.3 O3 leaderboard (11 complications, multi-label)

In [ ]:
def run_parallel_pipeline_o3(X_raw, y_comp, fold_ids, n_folds, verbose=True):
    """Scoped to Random Forest / Extra Trees multi-output wrappers, which were the only
    competitive algorithms for O3 in preliminary comparisons (see paper Section 4.1)."""
    base_estimators = {
        "RandomForest-MultiOutput": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
        "ExtraTrees-MultiOutput": ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    }
    n_comp = y_comp.shape[1]
    rows = []
    oof_proba = {name: np.zeros((len(X_raw), n_comp)) for name in base_estimators}

    for fold, (tr, vl) in enumerate(iter_fold_splits(fold_ids, n_folds), 1):
        imp, sc, Xtr = make_preprocessor(X_raw[tr])
        Xvl = apply_preprocessor(imp, sc, X_raw[vl])
        ytr, yvl = y_comp[tr], y_comp[vl]

        for name, base in base_estimators.items():
            model = MultiOutputClassifier(clone(base))
            model.fit(Xtr, ytr)
            proba_list = model.predict_proba(Xvl)

            p_comp = np.zeros((Xvl.shape[0], n_comp))
            for j, est in enumerate(model.estimators_):
                classes = list(est.classes_)
                if 1 in classes:
                    idx1 = classes.index(1)
                    p_comp[:, j] = proba_list[j][:, idx1]
                # else: label had zero positives in this training fold -> leave at 0
            oof_proba[name][vl] = p_comp

            aucs = []
            for j in range(n_comp):
                yj = yvl[:, j]
                if yj.sum() and (1 - yj).sum():
                    aucs.append(roc_auc_score(yj, p_comp[:, j]))
            macro_auc = float(np.mean(aucs)) if aucs else np.nan
            pred = (p_comp >= 0.5).astype(int)
            macro_f1 = f1_score(yvl, pred, average="macro", zero_division=0)
            micro_f1 = f1_score(yvl, pred, average="micro", zero_division=0)
            hamming = float(np.mean(pred != yvl))
            micro_auc = roc_auc_score(yvl.ravel(), p_comp.ravel())

            rows.append({
                "algorithm": name, "fold": fold, "macro_auc": macro_auc, "micro_auc": micro_auc,
                "macro_f1": macro_f1, "micro_f1": micro_f1, "hamming_loss": hamming,
            })

        if verbose:
            print(f"[Parallel Pipeline O3] fold {fold}/{n_folds} done")

    rows_df = pd.DataFrame(rows)
    leaderboard = (
        rows_df.groupby("algorithm").mean(numeric_only=True)
        .drop(columns=["fold"]).sort_values("macro_auc", ascending=False)
    )
    return rows_df, leaderboard, oof_proba

o3_pp_rows, o3_pp_leaderboard, o3_pp_oof_proba = run_parallel_pipeline_o3(
    X_raw, y_comp, GLOBAL_FOLD_ID, N_FOLDS, verbose=True
)
o3_pp_rows.to_csv(f"{SAVE_DIR}/parallel_pipeline_o3_per_fold.csv", index=False)
o3_pp_leaderboard.to_csv(f"{SAVE_DIR}/parallel_pipeline_o3_leaderboard.csv")
best_o3_name = o3_pp_leaderboard.index[0]
oof_parallel_p_comp = o3_pp_oof_proba[best_o3_name]
print("\nO3 leaderboard (mean across folds, ranked by macro-AUC):")
display(o3_pp_leaderboard.round(4))
print("Best O3 classical model:", best_o3_name)


## 13.4 Combine into the hierarchical formulation

Same hierarchical coupling used everywhere else in this notebook -- `P(death via cause k | X) = P(death | X) x P(cause k | death, X)` -- built from the best classical O1 and O2 models instead of an FTT.

In [ ]:
def hierarchical_probabilities_from_probs(p_dead, p_cause_probs):
    """Same hierarchical combination as `hierarchical_probabilities`, but for models
    that output probabilities directly (classical sklearn/xgboost/lightgbm models)
    rather than logits that need a softmax."""
    p_dead = np.clip(np.asarray(p_dead), 1e-7, 1 - 1e-7)
    row_sums = p_cause_probs.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1.0, row_sums)
    p_cause_norm = p_cause_probs / row_sums
    p_alive = 1.0 - p_dead
    p_fatal = p_dead[:, None] * p_cause_norm
    p8 = np.column_stack([p_alive, p_fatal])
    return p8, p_cause_norm, p_fatal

p8_pp, p_cause_norm_pp, p_fatal_pp = hierarchical_probabilities_from_probs(
    oof_parallel_p_dead, oof_parallel_p_cause
)

oof_parallel = {
    "p_dead": oof_parallel_p_dead,
    "p_cause_given_dead": p_cause_norm_pp,
    "p_fatal_cause": p_fatal_pp,
    "p_comp": oof_parallel_p_comp,
    "p8": p8_pp,
    "fold": GLOBAL_FOLD_ID.copy(),
}

print("Parallel Pipeline (best-of-leaderboard) assembled:")
print(f"  O1: {best_o1_name}")
print(f"  O2: {best_o2_name}")
print(f"  O3: {best_o3_name}")


## 13.5 Table 2 / Table 3: FT-Transformer vs. Parallel Pipeline

In [ ]:
# ================================================================
# Table 2 / Table 3 style comparison: FT-Transformer vs. Parallel Pipeline
# (admission-only features, same folds)
# ================================================================
ftt_o1_auc = all_results["full"]["o1_auc"].mean()
ftt_o3_auc = all_results["full"]["o3_macro_auc"].mean()
pp_o1_auc = o1_pp_leaderboard.loc[best_o1_name, "auc"]
pp_o3_auc = o3_pp_leaderboard.loc[best_o3_name, "macro_auc"]

table2 = pd.DataFrame([
    {"model": "Parallel classical pipeline", "feature_window": f"Admission ({P['n_features']})",
     "o1_auroc": pp_o1_auc, "o3_macro_auroc": pp_o3_auc},
    {"model": "FT-Transformer", "feature_window": f"Admission ({P['n_features']})",
     "o1_auroc": ftt_o1_auc, "o3_macro_auroc": ftt_o3_auc},
])
table2.to_csv(f"{SAVE_DIR}/table2_model_level_comparison.csv", index=False)
print("Table 2 -- model-level reference comparison (admission-only):")
display(table2.round(4))

# Per-complication AUC (best classical O3 model vs. FT-Transformer) -- Table 3 style.
ftt_oof = all_oof["full"]
comp_rows = []
for j, comp_name in enumerate(COMP_NAMES):
    yj = y_comp[:, j]
    ftt_auc = roc_auc_score(yj, ftt_oof["p_comp"][:, j]) if yj.sum() and (1 - yj).sum() else np.nan
    pp_auc = roc_auc_score(yj, oof_parallel_p_comp[:, j]) if yj.sum() and (1 - yj).sum() else np.nan
    comp_rows.append({"complication": comp_name, "ftt_admission_auc": ftt_auc, "parallel_pipeline_auc": pp_auc})
table3 = pd.DataFrame(comp_rows)
table3.loc["Macro mean"] = ["Macro mean", table3["ftt_admission_auc"].mean(), table3["parallel_pipeline_auc"].mean()]
table3.to_csv(f"{SAVE_DIR}/table3_per_complication_comparison.csv", index=False)
print("\\nTable 3 -- per-complication AUC comparison (admission-only):")
display(table3.round(4))


# Section 4.3 / Experiment 2: Output-Formulation Comparison

Tests three ways to produce the final 8-way mortality/cause-of-death outcome, using the same out-of-fold predictions and the same 5-fold splits:

- **Hard gate:** predict a cause only when `P(death) >= 0.50`; otherwise suppress cause.
- **Hierarchical (primary approach used in the paper):** `P(death) x P(cause | death, X)` for every patient, with no threshold.
- **Flat 8-class:** directly predict alive + 7 fatal causes with one multiclass FT-Transformer.

In [ ]:
# ================================================================
# Hard-gated evaluation on the fully-shared model's OOF hierarchical predictions
# ================================================================

def hard_gate_predictions(p_dead, p_cause_given_dead, y_true8, threshold=0.5):
    p_dead = np.asarray(p_dead)
    p_cause_given_dead = np.asarray(p_cause_given_dead)
    gate_dead = p_dead >= threshold
    pred8 = np.zeros(len(p_dead), dtype=int)
    pred8[gate_dead] = 1 + p_cause_given_dead[gate_dead].argmax(axis=1)

    actual_death = y_true8 > 0
    suppressed_deaths = actual_death & (~gate_dead)
    false_death_gate = (~actual_death) & gate_dead
    return {
        'pred8': pred8,
        'gate_dead': gate_dead,
        'suppressed_deaths': suppressed_deaths,
        'false_death_gate': false_death_gate,
        'n_suppressed_deaths': int(suppressed_deaths.sum()),
        'suppressed_death_rate': float(suppressed_deaths.sum() / max(actual_death.sum(), 1)),
        'false_cause_rate_survivors': float(false_death_gate.sum() / max((~actual_death).sum(), 1)),
    }

def hard_gate_metrics(p_dead, p_cause_given_dead, y_true8, threshold=0.5):
    h = hard_gate_predictions(p_dead, p_cause_given_dead, y_true8, threshold)
    pred8 = h['pred8']
    acc = accuracy_score(y_true8, pred8)
    macro_f1 = f1_score(y_true8, pred8, average='macro', zero_division=0)
    deaths = y_true8 > 0
    system_recall = ((pred8 == y_true8) & deaths).sum() / max(deaths.sum(), 1)
    cause_pred_mask = pred8 > 0
    correct_cause_given_pred = ((pred8 == y_true8) & cause_pred_mask).sum() / max(cause_pred_mask.sum(), 1)
    return {
        'threshold': threshold,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'system_recall_among_deaths': system_recall,
        'suppressed_death_rate': h['suppressed_death_rate'],
        'false_cause_rate_survivors': h['false_cause_rate_survivors'],
        'cause_precision_among_cause_predictions': correct_cause_given_pred,
        'n_suppressed_deaths': h['n_suppressed_deaths'],
    }

# The fully-shared model is the primary hierarchical model for this comparison.
PRIMARY_SHARING = 'full'
PRIMARY_OOF = all_oof[PRIMARY_SHARING]

thresholds = np.arange(0.10, 0.91, 0.05)
hard_rows = [
    hard_gate_metrics(
        PRIMARY_OOF['p_dead'],
        PRIMARY_OOF['p_cause_given_dead'],
        y_mort,
        threshold=float(t),
    )
    for t in thresholds
]
hard_gate_df = pd.DataFrame(hard_rows)
hard_gate_df.to_csv(f'{SAVE_DIR}/hard_gate_threshold_sensitivity.csv', index=False)
print('Hard-gating threshold sensitivity (full-sharing OOF predictions):')
display(hard_gate_df.round(4))


## 14.1 Direct comparison at the conventional 0.50 threshold

In [ ]:
# Hard-gated metrics at the conventional threshold, t=0.50
hard50 = hard_gate_metrics(
    PRIMARY_OOF['p_dead'],
    PRIMARY_OOF['p_cause_given_dead'],
    y_mort,
    threshold=0.50,
)

# Hierarchical metrics from the already-computed OOF 8-way probabilities (no threshold).
hier50 = hierarchical_metrics(PRIMARY_OOF['p8'], y_mort)

formulation_compare = pd.DataFrame([
    {
        'formulation': 'Hard gate (t=0.50)',
        'accuracy': hard50['accuracy'],
        'macro_f1': hard50['macro_f1'],
        'system_recall_among_deaths': hard50['system_recall_among_deaths'],
        'suppressed_death_rate': hard50['suppressed_death_rate'],
        'false_cause_rate_survivors': hard50['false_cause_rate_survivors'],
        'fatal_cause_auc': np.nan,
        'log_loss': np.nan,
    },
    {
        'formulation': 'Hierarchical probabilities',
        'accuracy': hier50['end_to_end_acc'],
        'macro_f1': hier50['end_to_end_macro_f1'],
        'system_recall_among_deaths': hier50['system_recall_among_deaths'],
        'suppressed_death_rate': 0.0,
        'false_cause_rate_survivors': np.nan,
        'fatal_cause_auc': hier50['end_to_end_mean_fatal_cause_auc'],
        'log_loss': hier50['end_to_end_log_loss'],
    },
])
formulation_compare.to_csv(f'{SAVE_DIR}/hard_gate_vs_hierarchical.csv', index=False)
display(formulation_compare.round(4))

print('Hard-gate error propagation at t=0.50:')
print(f"  Actual deaths suppressed by O1 false negatives: {hard50['n_suppressed_deaths']}")
print(f"  Suppressed-death rate: {hard50['suppressed_death_rate']:.4f}")
print(f"  Survivor false-cause rate: {hard50['false_cause_rate_survivors']:.4f}")


## 14.2 Flat 8-class baseline

Uses the same admission-only features, fold-safe preprocessing, 5-fold CV, FT-Transformer tokenization/backbone, and early-stopping framework. The only change from the primary model is replacing O1 + conditional O2 with a single 8-class head.

In [ ]:
# ================================================================
# Flat 8-class FT-Transformer baseline
# ================================================================
class Flat8Model(nn.Module):
    def __init__(self, p):
        super().__init__()
        d = p['d_token']
        self.tok, self.enc = make_encoder(p)
        self.head = TaskHead(d, 8, p['head_dropout'])

    def forward(self, x):
        z = run_encoder(self.tok, self.enc, x)
        return self.head(z)

def flat_class_weights(y_train, n_classes=8, cap=10.0):
    counts = np.bincount(y_train.astype(int), minlength=n_classes).astype(float)
    counts = np.clip(counts, 1.0, None)
    w = counts.sum() / (n_classes * counts)
    return torch.tensor(np.minimum(w, cap), dtype=torch.float32, device=DEVICE)

def flat_train_epoch(model, loader, optimizer, criterion):
    model.train(); total=0.0; n=0
    for Xb, _, ym, _ in loader:
        Xb, ym = Xb.to(DEVICE), ym.to(DEVICE)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, ym)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * Xb.size(0); n += Xb.size(0)
    return total / max(n,1)

@torch.no_grad()
def flat_predict(model, loader, criterion):
    model.eval(); losses=[]; probs=[]; ys=[]
    total=0.0; n=0
    for Xb, _, ym, _ in loader:
        Xb, ym = Xb.to(DEVICE), ym.to(DEVICE)
        logits=model(Xb)
        loss=criterion(logits,ym)
        total += loss.item()*Xb.size(0); n += Xb.size(0)
        probs.append(torch.softmax(logits,dim=1).cpu().numpy())
        ys.append(ym.cpu().numpy())
    return total/max(n,1), np.vstack(probs), np.concatenate(ys)

def flat_multiclass_metrics(p8, y8):
    pred = p8.argmax(axis=1)
    per_class_auc=[]
    for k in range(8):
        yk=(y8==k).astype(int)
        if yk.sum() and (1-yk).sum():
            per_class_auc.append(roc_auc_score(yk,p8[:,k]))
    return {
        'accuracy': accuracy_score(y8,pred),
        'macro_f1': f1_score(y8,pred,average='macro',zero_division=0),
        'macro_auc': float(np.mean(per_class_auc)),
        'log_loss': log_loss(y8,np.clip(p8,1e-8,1-1e-8),labels=np.arange(8)),
        'system_recall_among_deaths': float(((pred==y8)&(y8>0)).sum()/max((y8>0).sum(),1)),
    }

def run_flat_cv(X_raw, y_comp, y_mort, y_bin, p, fold_ids, n_folds, verbose=True):
    rows=[]; oof=np.zeros((len(X_raw),8),dtype=np.float32)
    for fold,(tr,vl) in enumerate(iter_fold_splits(fold_ids, n_folds),1):
        if verbose: print('\n'+'='*72); print(f'[flat8] FOLD {fold}/{n_folds}')
        imp,sc,Xtr=make_preprocessor(X_raw[tr]); Xvl=apply_preprocessor(imp,sc,X_raw[vl])
        tr_dl=DataLoader(MIDataset(Xtr,y_comp[tr],y_mort[tr],y_bin[tr]),batch_size=p['batch_size'],shuffle=True)
        vl_dl=DataLoader(MIDataset(Xvl,y_comp[vl],y_mort[vl],y_bin[vl]),batch_size=p['batch_size'],shuffle=False)
        weights=flat_class_weights(y_mort[tr],8,p['cause_weight_cap'])
        criterion=nn.CrossEntropyLoss(weight=weights)
        model=Flat8Model(p).to(DEVICE)
        optimizer=torch.optim.AdamW(model.parameters(),lr=p['lr'],weight_decay=p['weight_decay'])
        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=p['epochs'])
        stopper=EarlyStopping(p['patience'])
        for epoch in range(1,p['epochs']+1):
            tr_loss=flat_train_epoch(model,tr_dl,optimizer,criterion); scheduler.step()
            val_loss,_,_=flat_predict(model,vl_dl,criterion)
            if verbose and epoch%30==0: print(f'  epoch {epoch:3d}: train={tr_loss:.4f}, val={val_loss:.4f}')
            if stopper.step(val_loss,model):
                if verbose: print(f'  early stop at epoch {epoch}')
                break
        stopper.restore(model)
        _,pvl,yvl=flat_predict(model,vl_dl,criterion)
        oof[vl]=pvl
        m=flat_multiclass_metrics(pvl,yvl)
        if verbose: print(f"  AUC={m['macro_auc']:.4f} | macro-F1={m['macro_f1']:.4f} | log-loss={m['log_loss']:.4f} | system recall={m['system_recall_among_deaths']:.4f}")
        rows.append({'fold':fold,**m})
    return pd.DataFrame(rows),oof

flat_results, flat_oof = run_flat_cv(
    X_raw, y_comp, y_mort, y_binary, P,
    fold_ids=GLOBAL_FOLD_ID, n_folds=N_FOLDS, verbose=True,
)
flat_summary=flat_results.mean(numeric_only=True)
flat_results.to_csv(f'{SAVE_DIR}/flat8_cv_results.csv',index=False)
flat_summary.to_csv(f'{SAVE_DIR}/flat8_summary.csv')
print('\\nFlat 8-class 5-fold mean:')
print(flat_summary)


## 14.3 Hierarchical vs. flat 8-class comparison

In [ ]:
hier_rows=[]
for sharing in CONDITIONS:
    oof=all_oof[sharing]
    hm=hierarchical_metrics(oof['p8'],y_mort)
    hier_rows.append({
        'model':'Hierarchical FTT',
        'macro_auc':hm['end_to_end_mean_fatal_cause_auc'],
        'macro_f1':hm['end_to_end_macro_f1'],
        'accuracy':hm['end_to_end_acc'],
        'log_loss':hm['end_to_end_log_loss'],
        'system_recall_among_deaths':hm['system_recall_among_deaths'],
    })
flat_mean=flat_results.mean(numeric_only=True)
hier_rows.append({
    'model':'Flat 8-class',
    'macro_auc':flat_mean['macro_auc'],
    'macro_f1':flat_mean['macro_f1'],
    'accuracy':flat_mean['accuracy'],
    'log_loss':flat_mean['log_loss'],
    'system_recall_among_deaths':flat_mean['system_recall_among_deaths'],
})
formulation_table=pd.DataFrame(hier_rows)
formulation_table.to_csv(f'{SAVE_DIR}/hierarchical_vs_flat8.csv',index=False)
print('Table 5 -- cause-of-death output-formulation results (hierarchical vs. flat):')
display(formulation_table.round(4))


## 14.4 Table 5: hard gate vs. hierarchical vs. flat 8-class, combined

In [ ]:
# ================================================================
# Final Table 5: hard gate vs. hierarchical vs. flat 8-class, side by side
# ================================================================
table5 = pd.concat([
    formulation_compare.rename(columns={'fatal_cause_auc': 'macro_auc'})[
        ['formulation', 'macro_auc', 'macro_f1', 'accuracy', 'log_loss']
    ],
    formulation_table.rename(columns={'model': 'formulation'})[
        ['formulation', 'macro_auc', 'macro_f1', 'accuracy', 'log_loss']
    ],
], ignore_index=True)
table5.to_csv(f'{SAVE_DIR}/table5_formulation_comparison.csv', index=False)
print('Table 5 -- cause-of-death output-formulation results (all three formulations):')
display(table5.round(3))


# Section 4.2 / Experiment 1: Time-Window Ablation

Trains the primary architecture (`sharing="full"`) on each of the three additional time windows (24h, 48h, 72h). The admission-only result is already available from Section 12 above, so it is not retrained. This answers: how much does waiting for more of the hospital stay actually buy you, relative to predicting at the moment of admission?

In [ ]:
EXTRA_WINDOWS = ["24h", "48h", "72h"]  # "admission" already trained above

window_results = {"admission": all_results["full"]}
window_oof = {"admission": all_oof["full"]}
window_feature_names = {"admission": FEATURE_NAMES}
window_n_features = {"admission": P["n_features"]}

for window in EXTRA_WINDOWS:
    print("\n" + "#" * 72)
    print(f"# TIME WINDOW: {window} -- {WINDOW_LABELS[window]}")
    print("#" * 72)

    Xw, featw = load_window(window)
    Pw = dict(P)
    Pw["n_features"] = Xw.shape[1]
    print(f"  features: {Xw.shape[1]}")

    results_df, oof, n_params = run_cv(
        "full", Xw, y_comp, y_mort, y_binary, Pw,
        fold_ids=GLOBAL_FOLD_ID, n_folds=N_FOLDS, verbose=True,
    )
    window_results[window] = results_df
    window_oof[window] = oof
    window_feature_names[window] = featw
    window_n_features[window] = Xw.shape[1]
    results_df.to_csv(f"{SAVE_DIR}/cv_results_window_{window}.csv", index=False)

    print(f"\n[{window}] {N_FOLDS}-fold CV summary")
    print(results_df.mean(numeric_only=True))


## 15.1 Table 4: comparison table across time windows

In [ ]:
window_metric_cols = [
    "o1_auc", "o1_brier", "o3_macro_auc", "o3_macro_f1",
    "o2_conditional_auc_dead", "o2_conditional_f1_dead",
    "fatal_cause_auc_all", "end_to_end_macro_f1", "end_to_end_log_loss",
    "system_recall",
]

window_summary_rows = []
for window in TIME_WINDOWS:
    d = window_results[window]
    row = {"window": window, "label": WINDOW_LABELS[window], "n_features": window_n_features[window]}
    for m in window_metric_cols:
        row[f"{m}_mean"] = d[m].mean()
        row[f"{m}_std"] = d[m].std()
    window_summary_rows.append(row)

window_summary_df = pd.DataFrame(window_summary_rows).set_index("window")
window_summary_df.to_csv(f"{SAVE_DIR}/time_window_comparison_summary.csv")

display_rows = []
for window in TIME_WINDOWS:
    d = window_results[window]
    row = {"window": f"{WINDOW_LABELS[window]} ({window_n_features[window]} feats)"}
    for m in window_metric_cols:
        row[m] = f"{d[m].mean():.4f} +/- {d[m].std():.4f}"
    display_rows.append(row)
window_display_df = pd.DataFrame(display_rows).set_index("window")
print("Table 4 -- FT-Transformer performance across cumulative time windows:")
window_display_df.T


## 15.2 Figure 2: visualize the trend across windows

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
plot_metrics = [
    ("o1_auc", "O1 mortality AUROC"),
    ("fatal_cause_auc_all", "Fatal-pathway AUROC (joint O1 x O2)"),
    ("o2_conditional_auc_dead", "O2 cause-of-death AUROC (deceased only)"),
]
x = np.arange(len(TIME_WINDOWS))
for ax, (metric, title) in zip(axes, plot_metrics):
    means = [window_results[w][metric].mean() for w in TIME_WINDOWS]
    stds = [window_results[w][metric].std() for w in TIME_WINDOWS]
    ax.errorbar(x, means, yerr=stds, marker="o", capsize=4, color=ARCH_BLUE)
    ax.set_xticks(x)
    ax.set_xticklabels(TIME_WINDOWS)
    ax.set_title(title)
    ax.set_xlabel("Time window")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/time_window_trend.png", dpi=300, bbox_inches="tight")
plt.show()


# Section 4.4 / Experiment 3: SHAP Interpretability

Which admission-time features drive each outcome, and how stable are those drivers as more of the hospital stay becomes available? A separate fully-shared FT-Transformer is trained per time window (85% train / 15% internal validation for early stopping only -- these models are not used for the CV performance results above). Permutation SHAP uses an independent 200-patient background sample.

## 16.1 Train the final full-cohort FT-Transformer (admission window, interpretation model)

In [ ]:
from sklearn.model_selection import train_test_split

# Held-out slice used only for early stopping -- this model is never scored for
# CV performance; it exists purely to produce SHAP explanations (paper Section 3.5,
# Experiment 3).
idx_all = np.arange(len(X_raw))
idx_tr, idx_val = train_test_split(idx_all, test_size=0.15, random_state=SEED, stratify=y_binary)

IMP_FINAL, SC_FINAL, Xtr_final = make_preprocessor(X_raw[idx_tr])
Xval_final = apply_preprocessor(IMP_FINAL, SC_FINAL, X_raw[idx_val])

tr_dl_final = DataLoader(
    MIDataset(Xtr_final, y_comp[idx_tr], y_mort[idx_tr], y_binary[idx_tr]),
    batch_size=P["batch_size"], shuffle=True, drop_last=False,
)
vl_dl_final = DataLoader(
    MIDataset(Xval_final, y_comp[idx_val], y_mort[idx_val], y_binary[idx_val]),
    batch_size=P["batch_size"], shuffle=False,
)

loss_fn_final = make_loss(P, y_comp[idx_tr], y_mort[idx_tr], y_binary[idx_tr], DEVICE)
FINAL_FTT = FTTransformerSharing(P, sharing="full").to(DEVICE)
optimizer_final = torch.optim.AdamW(FINAL_FTT.parameters(), lr=P["lr"], weight_decay=P["weight_decay"])
scheduler_final = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_final, T_max=P["epochs"])
stopper_final = EarlyStopping(P["patience"])

for epoch in range(1, P["epochs"] + 1):
    train_loss = train_epoch(FINAL_FTT, tr_dl_final, optimizer_final, loss_fn_final)
    scheduler_final.step()
    val_loss, *_ = predict(FINAL_FTT, vl_dl_final, loss_fn_final)
    if epoch % 30 == 0:
        print(f"  epoch {epoch:3d}: train={train_loss:.4f}, val={val_loss:.4f}")
    if stopper_final.step(val_loss, FINAL_FTT):
        print(f"  early stop at epoch {epoch}")
        break
stopper_final.restore(FINAL_FTT)
FINAL_FTT.eval()
print(f"Final full-sharing FTT trained for SHAP interpretation "
      f"({idx_tr.size} train / {idx_val.size} internal-validation patients).")
print("SHAP compute device:", DEVICE, "-", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

# Same preprocessing statistics (fit on the training slice only) applied to every patient.
X_FULL_SCALED = apply_preprocessor(IMP_FINAL, SC_FINAL, X_raw)
X_FULL_DF = pd.DataFrame(X_FULL_SCALED, columns=FEATURE_NAMES)

@torch.no_grad()
def ftt_predict_batch(X_2d, batch_size=4096):
    """Black-box forward pass: numpy (n, n_features) scaled -> (p_dead, p_cause_given_dead, p_comp)."""
    FINAL_FTT.eval()
    X_2d = np.asarray(X_2d, dtype=np.float32)
    p_dead_all, p_cause_all, p_comp_all = [], [], []
    for start in range(0, len(X_2d), batch_size):
        Xb = torch.tensor(X_2d[start:start + batch_size]).to(DEVICE)
        o1, o2, o3 = FINAL_FTT(Xb)
        p_dead_all.append(torch.sigmoid(o1).squeeze(-1).cpu().numpy())
        p_cause_all.append(torch.softmax(o2, dim=1).cpu().numpy())
        p_comp_all.append(torch.sigmoid(o3).cpu().numpy())
    return np.concatenate(p_dead_all), np.concatenate(p_cause_all, axis=0), np.concatenate(p_comp_all, axis=0)

def make_o1_predict_fn():
    def f(X_2d):
        p_dead, _, _ = ftt_predict_batch(X_2d)
        return p_dead
    return f

def make_o2_predict_fn(cause_idx):
    def f(X_2d):
        _, p_cause, _ = ftt_predict_batch(X_2d)
        return p_cause[:, cause_idx]
    return f

def make_o3_predict_fn(comp_idx):
    def f(X_2d):
        _, _, p_comp = ftt_predict_batch(X_2d)
        return p_comp[:, comp_idx]
    return f

p_dead_all_ftt, p_cause_all_ftt, p_comp_all_ftt = ftt_predict_batch(X_FULL_SCALED)


## 16.2 O1 mortality: permutation SHAP (Figure 4a)

In [ ]:
!pip -q install -U shap
import shap
shap.initjs()

# The cohort is only ~1700 patients, so explain ALL of them rather than subsampling.
N_BACKGROUND = 200
N_EXPLAIN = len(X_FULL_DF)                   # full cohort
MIN_EVALS = 2 * P["n_features"] + 1          # permutation SHAP's required minimum
MAX_EVALS = 4 * MIN_EVALS                    # more permutations -> more stable SHAP estimates

rng = np.random.default_rng(SEED)
bg_idx = rng.choice(len(X_FULL_DF), size=min(N_BACKGROUND, len(X_FULL_DF)), replace=False)
background_o1 = X_FULL_DF.iloc[bg_idx]

masker_o1 = shap.maskers.Independent(background_o1, max_samples=N_BACKGROUND)
o1_explainer = shap.Explainer(make_o1_predict_fn(), masker_o1, feature_names=FEATURE_NAMES, algorithm="permutation")

explain_idx = rng.choice(len(X_FULL_DF), size=min(N_EXPLAIN, len(X_FULL_DF)), replace=False)
X_explain_o1 = X_FULL_DF.iloc[explain_idx].reset_index(drop=True)
o1_shap_values = o1_explainer(X_explain_o1, max_evals=MAX_EVALS)

plt.figure()
shap.plots.beeswarm(o1_shap_values, max_display=20, show=False, color=ARCH_DIVERGING_CMAP)
plt.title("O1 mortality (FT-Transformer) \u2014 SHAP summary (top 20 features)")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/shap_summary_o1_mortality_ftt.png", dpi=300, bbox_inches="tight")
plt.show()

o1_mean_abs_shap = np.abs(o1_shap_values.values).mean(axis=0)
o1_shap_rank = pd.Series(o1_mean_abs_shap, index=FEATURE_NAMES).sort_values(ascending=False)
o1_shap_rank.to_csv(f"{SAVE_DIR}/shap_importance_o1_mortality_ftt.csv", header=["mean_abs_shap"])
print("Top 10 features driving O1 mortality (mean |SHAP|, FT-Transformer) -- Table 6 (admission row):")
display(o1_shap_rank.head(10))


## 16.3 O2 (per cause of death) and O3 (per complication): permutation SHAP

In [ ]:
def explain_output(predict_fn, X_df, background_df, n_explain=200, seed=SEED):
    if len(X_df) < 10:
        return None
    rng_local = np.random.default_rng(seed)
    bg = background_df.iloc[rng_local.choice(len(background_df), size=min(N_BACKGROUND, len(background_df)), replace=False)]
    masker = shap.maskers.Independent(bg, max_samples=N_BACKGROUND)
    explainer = shap.Explainer(predict_fn, masker, feature_names=FEATURE_NAMES, algorithm="permutation")
    n = min(n_explain, len(X_df))
    idx = rng_local.choice(len(X_df), size=n, replace=False)
    X_sub = X_df.iloc[idx].reset_index(drop=True)
    sv = explainer(X_sub, max_evals=MAX_EVALS)
    mean_abs = np.abs(sv.values).mean(axis=0)
    return pd.Series(mean_abs, index=X_df.columns).sort_values(ascending=False)

N_EXPLAIN_PER_OUTCOME = len(X_FULL_DF)  # full cohort / full deceased sub-cohort per outcome

# --- O2: one permutation-SHAP model per cause of death, deceased sub-cohort only ---
dead_mask = y_binary == 1
X_dead_df = X_FULL_DF[dead_mask].reset_index(drop=True)
y_mort_dead = y_mort[dead_mask]

o2_shap_ranks = {}
for k, cname in enumerate(CAUSE_NAMES, start=1):
    y_k = (y_mort_dead == k).astype(int)
    if y_k.sum() < 5:
        print(f"[O2 SHAP] skipping '{cname}' \u2014 too few cases in the deceased sub-cohort")
        continue
    rank = explain_output(make_o2_predict_fn(k - 1), X_dead_df, X_dead_df, n_explain=N_EXPLAIN_PER_OUTCOME)
    if rank is not None:
        o2_shap_ranks[cname] = rank

# --- O3: one permutation-SHAP model per complication, full cohort ---
o3_shap_ranks = {}
for j, comp_name in enumerate(COMP_NAMES):
    y_j = y_comp[:, j].astype(int)
    if y_j.sum() < 5:
        print(f"[O3 SHAP] skipping '{comp_name}' \u2014 too few positive cases")
        continue
    rank = explain_output(make_o3_predict_fn(j), X_FULL_DF, X_FULL_DF, n_explain=N_EXPLAIN_PER_OUTCOME)
    if rank is not None:
        o3_shap_ranks[comp_name] = rank

top_feature_rows = []
for cname, rank in o2_shap_ranks.items():
    for feat, val in rank.head(10).items():
        top_feature_rows.append({"outcome_type": "cause_of_death", "outcome": cname,
                                  "feature": feat, "mean_abs_shap": val})
for comp_name, rank in o3_shap_ranks.items():
    for feat, val in rank.head(10).items():
        top_feature_rows.append({"outcome_type": "complication", "outcome": comp_name,
                                  "feature": feat, "mean_abs_shap": val})

top_features_by_outcome_df = pd.DataFrame(top_feature_rows)
top_features_by_outcome_df.to_csv(f"{SAVE_DIR}/shap_top_features_by_outcome_ftt.csv", index=False)
print("Top-10 SHAP features per cause of death and per complication (admission window):")
display(top_features_by_outcome_df)


In [ ]:
shap_summary = {
    "model_explained": "Fully-shared FT-Transformer, permutation SHAP, admission window",
    "o1_top10_features": o1_shap_rank.head(10).to_dict(),
    "o2_causes_modeled": list(o2_shap_ranks.keys()),
    "o3_complications_modeled": list(o3_shap_ranks.keys()),
}
with open(f"{SAVE_DIR}/shap_interpretability_summary_ftt.json", "w") as f:
    json.dump(shap_summary, f, indent=2)
print("Saved shap_interpretability_summary_ftt.json")
print(json.dumps(shap_summary, indent=2)[:1500])


## 16.4 SHAP across time windows (24h / 48h / 72h)

Repeats the full SHAP pipeline above for each additional time window, producing Figure 4b-d and the inputs to Tables 6-9.

In [ ]:
def jaccard(set_a, set_b):
    set_a, set_b = set(set_a), set(set_b)
    union = set_a | set_b
    if not union:
        return 1.0
    return len(set_a & set_b) / len(union)


def run_shap_analysis_for_window(window, Xw, featw):
    """Full SHAP pipeline for one time window: trains its own final full-cohort FTT
    on (Xw, featw), then computes O1 permutation SHAP and per-cause / per-complication
    permutation SHAP. Saves every output with a `_{window}` suffix and returns a dict
    of the key results for cross-window comparison (paper Section 4.4)."""
    print("\n" + "#" * 72)
    print(f"# SHAP ANALYSIS -- TIME WINDOW: {window} ({Xw.shape[1]} features)")
    print("#" * 72)

    Pw = dict(P)
    Pw["n_features"] = Xw.shape[1]

    idx_all = np.arange(len(Xw))
    idx_tr, idx_val = train_test_split(idx_all, test_size=0.15, random_state=SEED, stratify=y_binary)

    imp_w, sc_w, Xtr_w = make_preprocessor(Xw[idx_tr])
    Xval_w = apply_preprocessor(imp_w, sc_w, Xw[idx_val])

    tr_dl_w = DataLoader(MIDataset(Xtr_w, y_comp[idx_tr], y_mort[idx_tr], y_binary[idx_tr]),
                          batch_size=Pw["batch_size"], shuffle=True, drop_last=False)
    vl_dl_w = DataLoader(MIDataset(Xval_w, y_comp[idx_val], y_mort[idx_val], y_binary[idx_val]),
                          batch_size=Pw["batch_size"], shuffle=False)

    loss_fn_w = make_loss(Pw, y_comp[idx_tr], y_mort[idx_tr], y_binary[idx_tr], DEVICE)
    model_w = FTTransformerSharing(Pw, sharing="full").to(DEVICE)
    optimizer_w = torch.optim.AdamW(model_w.parameters(), lr=Pw["lr"], weight_decay=Pw["weight_decay"])
    scheduler_w = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_w, T_max=Pw["epochs"])
    stopper_w = EarlyStopping(Pw["patience"])

    for epoch in range(1, Pw["epochs"] + 1):
        train_epoch(model_w, tr_dl_w, optimizer_w, loss_fn_w)
        scheduler_w.step()
        val_loss, *_ = predict(model_w, vl_dl_w, loss_fn_w)
        if stopper_w.step(val_loss, model_w):
            break
    stopper_w.restore(model_w)
    model_w.eval()
    print(f"  Final full-sharing FTT trained ({idx_tr.size} train / {idx_val.size} internal-val patients)")

    X_full_scaled_w = apply_preprocessor(imp_w, sc_w, Xw)
    X_full_df_w = pd.DataFrame(X_full_scaled_w, columns=featw)

    @torch.no_grad()
    def predict_batch_w(X_2d, batch_size=4096):
        model_w.eval()
        X_2d = np.asarray(X_2d, dtype=np.float32)
        p_dead_all, p_cause_all, p_comp_all = [], [], []
        for start in range(0, len(X_2d), batch_size):
            Xb = torch.tensor(X_2d[start:start + batch_size]).to(DEVICE)
            o1, o2, o3 = model_w(Xb)
            p_dead_all.append(torch.sigmoid(o1).squeeze(-1).cpu().numpy())
            p_cause_all.append(torch.softmax(o2, dim=1).cpu().numpy())
            p_comp_all.append(torch.sigmoid(o3).cpu().numpy())
        return np.concatenate(p_dead_all), np.concatenate(p_cause_all, axis=0), np.concatenate(p_comp_all, axis=0)

    def o1_predict_fn_w(X_2d):
        p_dead, _, _ = predict_batch_w(X_2d)
        return p_dead

    def make_o2_fn_w(cause_idx):
        def f(X_2d):
            _, p_cause, _ = predict_batch_w(X_2d)
            return p_cause[:, cause_idx]
        return f

    def make_o3_fn_w(comp_idx):
        def f(X_2d):
            _, _, p_comp = predict_batch_w(X_2d)
            return p_comp[:, comp_idx]
        return f

    n_background = 200
    n_explain = len(X_full_df_w)
    min_evals = 2 * Pw["n_features"] + 1
    max_evals = 4 * min_evals

    rng_w = np.random.default_rng(SEED)
    bg_idx = rng_w.choice(len(X_full_df_w), size=min(n_background, len(X_full_df_w)), replace=False)
    background_w = X_full_df_w.iloc[bg_idx]
    masker_w = shap.maskers.Independent(background_w, max_samples=n_background)
    o1_explainer_w = shap.Explainer(o1_predict_fn_w, masker_w, feature_names=featw, algorithm="permutation")

    explain_idx = rng_w.choice(len(X_full_df_w), size=min(n_explain, len(X_full_df_w)), replace=False)
    X_explain_w = X_full_df_w.iloc[explain_idx].reset_index(drop=True)
    o1_shap_values_w = o1_explainer_w(X_explain_w, max_evals=max_evals)

    plt.figure()
    shap.plots.beeswarm(o1_shap_values_w, max_display=20, show=False, color=ARCH_DIVERGING_CMAP)
    plt.title(f"O1 mortality (FT-Transformer) \u2014 SHAP summary, {WINDOW_LABELS[window]}")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/shap_summary_o1_mortality_ftt_{window}.png", dpi=300, bbox_inches="tight")
    plt.show()

    o1_mean_abs_shap_w = np.abs(o1_shap_values_w.values).mean(axis=0)
    o1_shap_rank_w = pd.Series(o1_mean_abs_shap_w, index=featw).sort_values(ascending=False)
    o1_shap_rank_w.to_csv(f"{SAVE_DIR}/shap_importance_o1_mortality_ftt_{window}.csv", header=["mean_abs_shap"])
    print(f"  Top 10 O1 SHAP features ({window}):")
    display(o1_shap_rank_w.head(10))

    def explain_output_w(predict_fn, X_df, background_df, n_explain_local, seed=SEED):
        if len(X_df) < 10:
            return None
        rng_local = np.random.default_rng(seed)
        bg = background_df.iloc[rng_local.choice(len(background_df), size=min(n_background, len(background_df)), replace=False)]
        masker = shap.maskers.Independent(bg, max_samples=n_background)
        explainer = shap.Explainer(predict_fn, masker, feature_names=featw, algorithm="permutation")
        n = min(n_explain_local, len(X_df))
        idx = rng_local.choice(len(X_df), size=n, replace=False)
        X_sub = X_df.iloc[idx].reset_index(drop=True)
        sv = explainer(X_sub, max_evals=max_evals)
        mean_abs = np.abs(sv.values).mean(axis=0)
        return pd.Series(mean_abs, index=X_df.columns).sort_values(ascending=False)

    dead_mask_w = y_binary == 1
    X_dead_df_w = X_full_df_w[dead_mask_w].reset_index(drop=True)
    y_mort_dead_w = y_mort[dead_mask_w]

    o2_shap_ranks_w = {}
    for k, cname in enumerate(CAUSE_NAMES, start=1):
        y_k = (y_mort_dead_w == k).astype(int)
        if y_k.sum() < 5:
            print(f"[O2 SHAP] skipping '{cname}' -- too few cases in the deceased sub-cohort")
            continue
        rank = explain_output_w(make_o2_fn_w(k - 1), X_dead_df_w, X_dead_df_w, n_explain)
        if rank is not None:
            o2_shap_ranks_w[cname] = rank

    o3_shap_ranks_w = {}
    for j, comp_name in enumerate(COMP_NAMES):
        y_j = y_comp[:, j].astype(int)
        if y_j.sum() < 5:
            print(f"[O3 SHAP] skipping '{comp_name}' -- too few positive cases")
            continue
        rank = explain_output_w(make_o3_fn_w(j), X_full_df_w, X_full_df_w, n_explain)
        if rank is not None:
            o3_shap_ranks_w[comp_name] = rank

    top_rows_w = []
    for cname, rank in o2_shap_ranks_w.items():
        for feat, val in rank.head(10).items():
            top_rows_w.append({"outcome_type": "cause_of_death", "outcome": cname, "feature": feat, "mean_abs_shap": val})
    for comp_name, rank in o3_shap_ranks_w.items():
        for feat, val in rank.head(10).items():
            top_rows_w.append({"outcome_type": "complication", "outcome": comp_name, "feature": feat, "mean_abs_shap": val})
    top_by_outcome_w = pd.DataFrame(top_rows_w)
    top_by_outcome_w.to_csv(f"{SAVE_DIR}/shap_top_features_by_outcome_ftt_{window}.csv", index=False)

    summary_w = {
        "window": window, "n_features": Pw["n_features"],
        "o1_top10_features": o1_shap_rank_w.head(10).to_dict(),
    }
    with open(f"{SAVE_DIR}/shap_interpretability_summary_ftt_{window}.json", "w") as f:
        json.dump(summary_w, f, indent=2)

    return {
        "window": window, "o1_shap_rank": o1_shap_rank_w,
        "o2_shap_ranks": o2_shap_ranks_w, "o3_shap_ranks": o3_shap_ranks_w,
        "top_by_outcome": top_by_outcome_w,
    }


In [ ]:
EXTRA_SHAP_WINDOWS = ["24h", "48h", "72h"]  # admission SHAP already computed above

window_shap_results = {
    "admission": {
        "window": "admission", "o1_shap_rank": o1_shap_rank,
        "o2_shap_ranks": o2_shap_ranks, "o3_shap_ranks": o3_shap_ranks,
        "top_by_outcome": top_features_by_outcome_df,
    }
}

for window in EXTRA_SHAP_WINDOWS:
    Xw, featw = load_window(window)
    window_shap_results[window] = run_shap_analysis_for_window(window, Xw, featw)


## 16.5 Table 6: how the top O1 drivers change across time windows, and adjacent-window Jaccard stability

In [ ]:
# ================================================================
# Table 6: five highest-ranked O1 features by window, plus Table-6/Fig-4-style
# top-10 stability (Jaccard overlap between adjacent windows)
# ================================================================
top5_rows = []
for window in TIME_WINDOWS:
    rank = window_shap_results[window]["o1_shap_rank"]
    top5_rows.append({"window": WINDOW_LABELS[window],
                       "top_5_features": ", ".join(rank.head(5).index.tolist())})
table6 = pd.DataFrame(top5_rows)
table6.to_csv(f"{SAVE_DIR}/table6_o1_top5_by_window.csv", index=False)
print("Table 6 -- most prominent O1 mortality features across cumulative windows:")
display(table6)

comparison_rows = []
for window in TIME_WINDOWS:
    rank = window_shap_results[window]["o1_shap_rank"]
    for i, (feat, val) in enumerate(rank.head(10).items(), start=1):
        comparison_rows.append({"window": window, "rank": i, "feature": feat, "mean_abs_shap": val})

o1_top_features_by_window_df = pd.DataFrame(comparison_rows)
o1_top_features_by_window_df.to_csv(f"{SAVE_DIR}/o1_top_features_by_time_window.csv", index=False)

pivot = o1_top_features_by_window_df.pivot(index="rank", columns="window", values="feature")
pivot = pivot[TIME_WINDOWS]
print("\\nTop-10 O1 SHAP features by time window (rank 1 = strongest):")
display(pivot)

# Feature-set stability between consecutive windows (paper: 0.667 / 0.538 / 0.818).
stability_rows = []
for i in range(len(TIME_WINDOWS) - 1):
    a, b = TIME_WINDOWS[i], TIME_WINDOWS[i + 1]
    set_a = set(window_shap_results[a]["o1_shap_rank"].head(10).index)
    set_b = set(window_shap_results[b]["o1_shap_rank"].head(10).index)
    stability_rows.append({"from_window": a, "to_window": b, "top10_jaccard_overlap": jaccard(set_a, set_b)})
stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv(f"{SAVE_DIR}/o1_top_feature_stability_across_windows.csv", index=False)
print("\\nTop-10 feature-set overlap between consecutive time windows (1.0 = identical set):")
display(stability_df)


## 16.6 Tables 7-9: top-3 SHAP features per O2 cause and per O3 complication, by window

In [ ]:
# ================================================================
# Table 7 (O2 cause of death) and Tables 8-9 (O3 complications):
# three highest-ranked SHAP features for every outcome, at every window.
# ================================================================
def top3_pivot(outcome_names, rank_key):
    rows = {}
    for name in outcome_names:
        row = {}
        for window in TIME_WINDOWS:
            ranks = window_shap_results[window][rank_key]
            if name in ranks:
                row[window] = ", ".join(ranks[name].head(3).index.tolist())
            else:
                row[window] = "(too few cases)"
        rows[name] = row
    return pd.DataFrame(rows).T[TIME_WINDOWS]

table7 = top3_pivot(CAUSE_NAMES, "o2_shap_ranks")
table7.to_csv(f"{SAVE_DIR}/table7_o2_top3_by_window.csv")
print("Table 7 -- three highest-ranked SHAP features per O2 cause at each window:")
display(table7)

table8_9 = top3_pivot(COMP_NAMES, "o3_shap_ranks")
table8_9.to_csv(f"{SAVE_DIR}/table8_9_o3_top3_by_window.csv")
print("\\nTables 8-9 -- three highest-ranked SHAP features per O3 complication at each window:")
display(table8_9)


## 17. Download all final results

In [ ]:
## Save and inspect final results

from pathlib import Path

result_files = sorted(Path(SAVE_DIR).glob("**/*"))
result_files = [p for p in result_files if p.is_file()]
print(f"Saved {len(result_files)} result file(s) to {SAVE_DIR}")
for p in result_files:
    print(" -", p)
